# Scenario construction reference for cell-line-specific SLBench

This notebook is a cleaned reference for rebuilding the scenario splits under `data/SLbench/Scenario/Cell_line_specific`.

## Scope

- Read source benchmark files from `data/SLbench/Source`.
- Build scenario-specific train/test splits for each cell line.
- Save outputs to `data/SLbench/Scenario/Cell_line_specific/<Scenario>/<CellLine>/`.

## Notes

- Keep all `RUN_*` flags set to `False` unless you really want to regenerate files.
- The path configuration cell searches upward from the current working directory, so the notebook can be run from the repository root or from `tutorials/`.
- The current repository already contains generated scenario files, so this notebook is intentionally provided as a reference template and is not executed here.
- Bench columns follow the existing SLBench convention: `0 = gene1_primekg_index`, `1 = gene2_primekg_index`, `2 = cell_id`, `3 = label`.


## Reference scenario coverage in this repository

This reference notebook mirrors the current scenario organization:

- `Random_splitting`: 5 stratified folds for all cell lines listed in `data/SLbench/Source/cell_line.txt`
- `Cold_start`: 5 repeated node-disjoint splits for selected cell lines
- `Cross_functional`: 1 GO-module split for the cell lines with enrichment tables
- `Tail_node`: 1 head-vs-tail node split based on node frequency

If you want to extend the coverage later, update the scenario-specific cell lists in the guarded execution cells.


In [ ]:
from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold


def find_existing_path(*parts: str) -> Path:
    """Search upward from the current working directory until the requested path exists."""
    for root in (Path.cwd(), *Path.cwd().parents):
        candidate = root.joinpath(*parts)
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not locate path: {Path(*parts)}')


SOURCE_ROOT = find_existing_path('data', 'SLbench', 'Source')
OUTPUT_ROOT = find_existing_path('data', 'SLbench', 'Scenario', 'Cell_line_specific')
CELL_LIST_PATH = SOURCE_ROOT / 'cell_line.txt'
PROTEIN_MAP_PATH = find_existing_path(
    'data',
    'MultiOmics_feature',
    'cell_line_data',
    'protein_csv',
    'Multi_6_cell_lines_proteins.csv',
)
ENRICHMENT_ROOT = OUTPUT_ROOT / 'Cross_functional' / 'enrichment_results'

BENCH_COLUMNS = [0, 1, 2, 3]
DEFAULT_RANDOM_STATE = 42

EXPECTED_SCENARIO_LAYOUT = {
    'Random_splitting': ['22Rv1', 'A375', 'A549', 'Jurkat', 'MeWo', 'Pk1'],
    'Cold_start': ['22Rv1', 'A549', 'Jurkat', 'MeWo', 'Pk1'],
    'Cross_functional': ['A549', 'MeWo', 'Pk1'],
    'Tail_node': ['A549', 'Pk1'],
}

print(f'Using source root: {SOURCE_ROOT}')
print(f'Using scenario root: {OUTPUT_ROOT}')
print(f'Using protein map: {PROTEIN_MAP_PATH}')


In [ ]:
def read_cell_line_list(cell_list_path: Path = CELL_LIST_PATH) -> list[str]:
    if not cell_list_path.exists():
        raise FileNotFoundError(f'Cell line list not found: {cell_list_path}')

    return [line.strip() for line in cell_list_path.read_text().splitlines() if line.strip()]


def find_bench_path(cell_name: str, source_root: Path = SOURCE_ROOT) -> Path:
    cell_dir = source_root / cell_name
    if not cell_dir.exists():
        raise FileNotFoundError(f'Cell directory not found: {cell_dir}')

    bench_candidates = sorted(cell_dir.glob('*_bench.csv'))
    if len(bench_candidates) != 1:
        raise FileNotFoundError(
            f'Expected exactly one *_bench.csv file in {cell_dir}, found {len(bench_candidates)}'
        )

    return bench_candidates[0]


def load_bench_dataframe(cell_name: str, source_root: Path = SOURCE_ROOT) -> pd.DataFrame:
    bench_path = find_bench_path(cell_name, source_root=source_root)
    bench_df = pd.read_csv(bench_path)
    bench_df.columns = BENCH_COLUMNS
    return bench_df


def resolve_case_insensitive_dir(parent: Path, desired_name: str) -> Path:
    if parent.exists():
        for child in parent.iterdir():
            if child.is_dir() and child.name.lower() == desired_name.lower():
                return child
    return parent / desired_name


def prepare_output_dir(scenario_name: str, cell_name: str, output_root: Path = OUTPUT_ROOT) -> Path:
    scenario_root = output_root / scenario_name
    scenario_root.mkdir(parents=True, exist_ok=True)

    cell_dir = resolve_case_insensitive_dir(scenario_root, cell_name)
    cell_dir.mkdir(parents=True, exist_ok=True)
    return cell_dir


def save_split_files(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    scenario_name: str,
    cell_name: str,
    split_id: int,
    output_root: Path = OUTPUT_ROOT,
) -> Path:
    cell_dir = prepare_output_dir(scenario_name, cell_name, output_root=output_root)
    train_df.to_csv(cell_dir / f'sl_train_{split_id}.csv', index=False)
    test_df.to_csv(cell_dir / f'sl_test_{split_id}.csv', index=False)
    return cell_dir


def summarize_split(train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict:
    return {
        'train_rows': int(train_df.shape[0]),
        'test_rows': int(test_df.shape[0]),
        'train_labels': train_df[3].value_counts().to_dict(),
        'test_labels': test_df[3].value_counts().to_dict(),
    }


## Scenario 1: Random splitting

This scenario creates 5 stratified folds for each benchmark file.

- Input: one `*_bench.csv` file per cell line in `data/SLbench/Source/<CellLine>/`
- Output: `data/SLbench/Scenario/Cell_line_specific/Random_splitting/<CellLine>/sl_train_<fold>.csv` and `sl_test_<fold>.csv`
- Strategy: standard `StratifiedKFold` on the label column


In [ ]:
def build_random_splitting(
    cell_names: Sequence[str],
    n_splits: int = 5,
    random_state: int = DEFAULT_RANDOM_STATE,
    source_root: Path = SOURCE_ROOT,
    output_root: Path = OUTPUT_ROOT,
) -> dict[str, list[dict]]:
    results = {}

    for cell_name in cell_names:
        bench_df = load_bench_dataframe(cell_name, source_root=source_root)
        features = bench_df[[0, 1, 2]]
        labels = bench_df[3]
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

        fold_summaries = []
        for fold_id, (train_index, test_index) in enumerate(splitter.split(features, labels)):
            train_df = bench_df.iloc[train_index].reset_index(drop=True)
            test_df = bench_df.iloc[test_index].reset_index(drop=True)
            save_split_files(train_df, test_df, 'Random_splitting', cell_name, fold_id, output_root=output_root)
            fold_summaries.append({'fold': fold_id, **summarize_split(train_df, test_df)})

        results[cell_name] = fold_summaries

    return results


RUN_RANDOM_SPLITTING = False
RANDOM_SPLITTING_CELLS = read_cell_line_list()

if RUN_RANDOM_SPLITTING:
    random_results = build_random_splitting(RANDOM_SPLITTING_CELLS)
    random_results


## Scenario 2: Cold start

This scenario builds node-disjoint splits.

- A node is assigned to either the train partition or the test partition.
- A pair is kept only when **both** endpoints stay inside the same partition.
- The notebook repeats the split 5 times with different random seeds.
- The shuffle implementation intentionally follows the legacy notebook behavior: `np.random.seed(...)` followed by `np.random.shuffle(...)`.

This is useful for evaluating how the model generalizes to unseen genes instead of unseen edges only.


In [ ]:
def cold_start_split(
    bench_df: pd.DataFrame,
    node_col1: int = 0,
    node_col2: int = 1,
    test_ratio: float = 0.4,
    random_state: int = DEFAULT_RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    all_nodes = pd.concat([bench_df[node_col1], bench_df[node_col2]]).unique()
    np.random.seed(random_state)
    np.random.shuffle(all_nodes)

    test_node_count = int(len(all_nodes) * test_ratio)
    test_nodes = set(all_nodes[:test_node_count])
    train_nodes = set(all_nodes[test_node_count:])

    train_mask = bench_df[node_col1].isin(train_nodes) & bench_df[node_col2].isin(train_nodes)
    test_mask = bench_df[node_col1].isin(test_nodes) & bench_df[node_col2].isin(test_nodes)

    train_df = bench_df[train_mask].reset_index(drop=True)
    test_df = bench_df[test_mask].reset_index(drop=True)
    return train_df, test_df


def build_cold_start(
    cell_names: Sequence[str],
    repeats: int = 5,
    test_ratio: float = 0.4,
    start_seed: int = 1024,
    source_root: Path = SOURCE_ROOT,
    output_root: Path = OUTPUT_ROOT,
) -> dict[str, list[dict]]:
    results = {}

    for cell_name in cell_names:
        bench_df = load_bench_dataframe(cell_name, source_root=source_root)
        split_summaries = []

        for split_id in range(repeats):
            train_df, test_df = cold_start_split(
                bench_df,
                test_ratio=test_ratio,
                random_state=start_seed + split_id,
            )
            save_split_files(train_df, test_df, 'Cold_start', cell_name, split_id, output_root=output_root)
            split_summaries.append({'split': split_id, **summarize_split(train_df, test_df)})

        results[cell_name] = split_summaries

    return results


RUN_COLD_START = False
COLD_START_CELLS = ['22Rv1', 'A549', 'Jurkat', 'MeWo', 'Pk1']

if RUN_COLD_START:
    cold_start_results = build_cold_start(COLD_START_CELLS)
    cold_start_results


## Scenario 3: Cross-functional split

This scenario uses precomputed enrichment tables to split benchmark pairs by biological function modules.

- Source benchmark pairs are loaded from `data/SLbench/Source`.
- GO module enrichment tables are read from `data/SLbench/Scenario/Cell_line_specific/Cross_functional/enrichment_results`.
- A repository-local index-to-gene-name map is built from `data/MultiOmics_feature/cell_line_data/protein_csv/Multi_6_cell_lines_proteins.csv`.
- The implementation below intentionally follows the legacy notebook behavior for module sampling, duplicate removal, overlap filtering, and `.npy` artifact naming.

One split is generated for each configured cell line.


In [ ]:
def load_index_to_gene_name_map(protein_map_path: Path = PROTEIN_MAP_PATH) -> dict[int, str]:
    if not protein_map_path.exists():
        raise FileNotFoundError(f'Protein map not found: {protein_map_path}')

    protein_df = pd.read_csv(protein_map_path)
    protein_df = protein_df[['primekg_index', 'protein_name']].dropna().drop_duplicates('primekg_index')
    protein_df['primekg_index'] = protein_df['primekg_index'].astype(int)
    return protein_df.set_index('primekg_index')['protein_name'].to_dict()


def find_enrichment_table(
    cell_name: str,
    enrichment_root: Path = ENRICHMENT_ROOT,
    top_k: int = 100,
) -> Path:
    if not enrichment_root.exists():
        raise FileNotFoundError(f'Enrichment directory not found: {enrichment_root}')

    expected_suffix = f'_GO_function_modules_top{top_k}.csv'
    for csv_path in sorted(enrichment_root.glob(f'*{expected_suffix}')):
        if csv_path.name.lower().startswith(cell_name.lower()):
            return csv_path

    raise FileNotFoundError(
        f'No enrichment table found for {cell_name} with suffix {expected_suffix} in {enrichment_root}'
    )


def get_cross_functional_prefix(enrichment_path: Path, fallback_cell_name: str) -> str:
    marker = '_GO_function_modules_top'
    if marker in enrichment_path.stem:
        return enrichment_path.stem.split(marker)[0]
    return fallback_cell_name


def normalize_pair(row: pd.Series) -> tuple[str, str]:
    return tuple(sorted([row['g1'], row['g2']]))


def cross_functional_split(
    bench_df: pd.DataFrame,
    module_df: pd.DataFrame,
    index_to_gene_name: dict[int, str],
    top_k: int = 100,
    random_state: int = DEFAULT_RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray]:
    annotated_df = bench_df.copy()
    annotated_df['g1'] = annotated_df[0].map(index_to_gene_name)
    annotated_df['g2'] = annotated_df[1].map(index_to_gene_name)

    bp_list = module_df['Module'].unique().tolist()
    np.random.seed(random_state)
    train_bp = np.random.choice(bp_list, size=int(top_k / 2), replace=False)
    test_bp = list(set(bp_list) - set(train_bp))

    train_genes = module_df[module_df['Module'].isin(train_bp)]['Gene'].unique().tolist()
    test_genes = module_df[module_df['Module'].isin(test_bp)]['Gene'].unique().tolist()

    train_df = annotated_df[annotated_df['g1'].isin(train_genes) & annotated_df['g2'].isin(train_genes)].copy()
    test_df = annotated_df[annotated_df['g1'].isin(test_genes) & annotated_df['g2'].isin(test_genes)].copy()

    train_df = train_df.drop_duplicates(subset=['g1', 'g2'])
    test_df = test_df.drop_duplicates(subset=['g1', 'g2'])

    train_df['pair'] = train_df.apply(normalize_pair, axis=1)
    test_df['pair'] = test_df.apply(normalize_pair, axis=1)
    test_df = test_df[~test_df['pair'].isin(set(train_df['pair']))]

    train_df.reset_index(inplace=True, drop=True)
    test_df.reset_index(inplace=True, drop=True)
    train_df = train_df[[0, 1, 2, 3]]
    test_df = test_df[[0, 1, 2, 3]]

    return train_df, test_df, np.array(train_bp), np.array(test_bp)


def build_cross_functional(
    cell_names: Sequence[str],
    top_k: int = 100,
    random_state: int = DEFAULT_RANDOM_STATE,
    source_root: Path = SOURCE_ROOT,
    output_root: Path = OUTPUT_ROOT,
    enrichment_root: Path = ENRICHMENT_ROOT,
    protein_map_path: Path = PROTEIN_MAP_PATH,
) -> dict[str, dict]:
    index_to_gene_name = load_index_to_gene_name_map(protein_map_path=protein_map_path)
    results = {}

    for cell_name in cell_names:
        bench_df = load_bench_dataframe(cell_name, source_root=source_root)
        enrichment_path = find_enrichment_table(cell_name, enrichment_root=enrichment_root, top_k=top_k)
        module_df = pd.read_csv(enrichment_path)
        train_df, test_df, train_bp, test_bp = cross_functional_split(
            bench_df,
            module_df,
            index_to_gene_name=index_to_gene_name,
            top_k=top_k,
            random_state=random_state,
        )

        cell_dir = save_split_files(train_df, test_df, 'Cross_functional', cell_name, 0, output_root=output_root)
        prefix = get_cross_functional_prefix(enrichment_path, fallback_cell_name=cell_name)
        np.save(cell_dir / f'{prefix}_train_bp.npy', train_bp)
        # Legacy-compatible save behavior: the original notebook wrote the train module array to both files.
        np.save(cell_dir / f'{prefix}_test_bp.npy', train_bp)

        results[cell_name] = {
            **summarize_split(train_df, test_df),
            'train_module_count': int(len(train_bp)),
            'test_module_count': int(len(test_bp)),
        }

    return results


RUN_CROSS_FUNCTIONAL = False
CROSS_FUNCTIONAL_CELLS = ['A549', 'MeWo', 'Pk1']

if RUN_CROSS_FUNCTIONAL:
    cross_functional_results = build_cross_functional(CROSS_FUNCTIONAL_CELLS)
    cross_functional_results


## Scenario 4: Tail-node split

This scenario reproduces the legacy tail-node construction logic.

- The node table is built from `list(set(df[0]) | set(df[1]))`, exactly as in the original notebook.
- The first `int(node_count * ratio)` nodes are assigned to the train-side node pool.
- The remaining nodes after index `tail_ratio + 1` are assigned to the test-side node pool.
- One node is therefore skipped between the two pools, matching the original slicing behavior.
- After splitting, the saved `cell_id` column is overwritten with `0`, again matching the legacy notebook.

Because this implementation depends on Python set iteration order, the exact node membership can vary across environments even when the code is unchanged.


In [ ]:
def compute_node_pairs(
    bench_df: pd.DataFrame,
    index_to_gene_name: dict[int, str],
) -> pd.DataFrame:
    nodep = {'name': [], 'node': [], 'pos': [], 'neg': [], 'sum': []}
    unique_nodes = list(set(bench_df[0]) | set(bench_df[1]))
    nodep['node'] = unique_nodes

    for node in unique_nodes:
        subdf = bench_df[(bench_df[0] == node) | (bench_df[1] == node)]
        count_value = subdf[3].value_counts()
        pos_count = count_value.get(1.0, 0)
        neg_count = count_value.get(0.0, 0)
        nodep['pos'].append(pos_count)
        nodep['neg'].append(neg_count)
        nodep['sum'].append(pos_count + neg_count)
        nodep['name'].append(index_to_gene_name.get(node, str(node)))

    return pd.DataFrame(nodep)


def construction_tail_ex(
    dfnode: pd.DataFrame,
    data_path: Path,
    ratio: float,
    store_path=None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    tail_ratio = int(dfnode.shape[0] * ratio)
    cellnode = dfnode.iloc[:tail_ratio]['node'].values
    tenode = dfnode.iloc[tail_ratio + 1:]['node'].values

    bench_df = pd.read_csv(data_path)
    bench_df.columns = BENCH_COLUMNS
    train_df = bench_df[(bench_df[0].isin(cellnode)) & (bench_df[1].isin(cellnode))].reset_index(drop=True)
    test_df = bench_df[(bench_df[0].isin(tenode)) & (bench_df[1].isin(tenode))].reset_index(drop=True)
    return train_df, test_df


def build_tail_node(
    cell_names: Sequence[str],
    ratio: float = 0.7,
    source_root: Path = SOURCE_ROOT,
    output_root: Path = OUTPUT_ROOT,
    protein_map_path: Path = PROTEIN_MAP_PATH,
) -> dict[str, dict]:
    index_to_gene_name = load_index_to_gene_name_map(protein_map_path=protein_map_path)
    results = {}

    for cell_name in cell_names:
        bench_path = find_bench_path(cell_name, source_root=source_root)
        bench_df = pd.read_csv(bench_path)
        bench_df.columns = BENCH_COLUMNS

        node_df = compute_node_pairs(bench_df, index_to_gene_name=index_to_gene_name)
        cell_dir = prepare_output_dir('Tail_node', cell_name, output_root=output_root)
        train_df, test_df = construction_tail_ex(node_df, data_path=bench_path, ratio=ratio, store_path=cell_dir)

        train_df = train_df.copy()
        test_df = test_df.copy()
        train_df[2] = 0
        test_df[2] = 0

        save_split_files(train_df, test_df, 'Tail_node', cell_name, 0, output_root=output_root)
        results[cell_name] = {
            **summarize_split(train_df, test_df),
            'node_count': int(node_df.shape[0]),
            'ratio': ratio,
        }

    return results


RUN_TAIL_NODE = False
TAIL_NODE_CELLS = ['A549', 'Pk1']

if RUN_TAIL_NODE:
    tail_node_results = build_tail_node(TAIL_NODE_CELLS)
    tail_node_results


## Optional layout check after regeneration

The helper below is only for a quick manual check after you deliberately regenerate files.
It is also guarded so the notebook remains safe to open without rewriting anything.


In [ ]:
def summarize_saved_layout(output_root: Path = OUTPUT_ROOT) -> dict[str, dict[str, list[str]]]:
    summary = {}
    for scenario_name in sorted(EXPECTED_SCENARIO_LAYOUT):
        scenario_root = output_root / scenario_name
        if not scenario_root.exists():
            summary[scenario_name] = {}
            continue

        cell_summary = {}
        for cell_dir in sorted([path for path in scenario_root.iterdir() if path.is_dir()], key=lambda path: path.name.lower()):
            cell_summary[cell_dir.name] = sorted(file_path.name for file_path in cell_dir.iterdir())
        summary[scenario_name] = cell_summary

    return summary


RUN_LAYOUT_CHECK = False

if RUN_LAYOUT_CHECK:
    summarize_saved_layout()
